In [1]:
import wandb
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np

api = wandb.Api()

# NT Benchmark scores from the paper, expressed as mean and variability
paper_nt_benchmark_scores = {
    "H3": (81.50, 0.80),
    "H3K4me1": (56.60, 1.01),
    "H3K4me2": (57.45, 2.27),
    "H3K4me3": (67.15, 0.93),
    "H3K9ac": (68.10, 1.91),
    "H3K14ac": (70.71, 2.32),
    "H3K36me3": (68.31, 1.19),
    "H3K79me3": (72.08, 1.23),
    "H4": (81.12, 0.93),
    "H4ac": (66.10, 1.20),
    "enhancers": (57.60, 2.52),
    "enhancers_types": (49.75, 2.82),
    "promoter_all": (96.82, 0.22),
    "promoter_no_tata": (96.76, 0.21),
    "promoter_tata": (96.34, 0.38),
    "splice_sites_acceptors": (96.23, 0.41),
    "splice_sites_donors": (96.62, 0.61),
    "splice_sites_all": (96.33, 0.31),
}

runs = api.runs("pipe-verri-gh/Canonicalization", filters={
    "jobType": "nt_benchmark_old",
    "group": "paper_baseline"
})

def build_runs_scores(runs, metrics):
    runs_scores = {}
    runs_metrics = {}
    for r in runs:
        dataset_name = r.config["dataset"]["dataset_name"]
        for metric in metrics:
            score = r.summary.get(metric)
            if score is not None:
                runs_metrics[dataset_name] = metric
                score = r.summary["best_score"]
                break
        
        if score is None:
            raise ValueError(f"Run {r.name} does not have any of the metrics {metrics} in its summary.")
        
        runs_scores[dataset_name] = score
    return runs_scores, runs_metrics

runs_scores, runs_metrics = build_runs_scores(runs, ["val/f1_macro", "val/mcc"])

In [3]:
def report_deviations(runs_scores, paper_scores, score_scale=1.0):
    """
    Print which tasks are above / below / within the paper's confidence interval
    (mean ± ci), showing each run score next to the paper's CI.

    runs_scores : dict {dataset_name: score} (output of build_runs_scores)
    paper_scores: dict {dataset_name: (mean, ci)}
    score_scale : multiply run scores by this before comparing
                  (use 100 if W&B logs fractions like 0.81 but the paper uses 81.5)
    """
    better, worse, within, missing = [], [], [], []

    for name, (mean, ci) in paper_scores.items():
        if name not in runs_scores:
            missing.append(name)
            continue

        score = runs_scores[name] * score_scale
        low, high = mean - ci, mean + ci
        row = (name, score, low, high, score - mean)

        if score > high:
            better.append(row)
        elif score < low:
            worse.append(row)
        else:
            within.append(row)

    def show(title, rows, reverse=False):
        print(f"=== {title}: {len(rows)} ===")
        for name, score, low, high, diff in sorted(rows, key=lambda r: r[4], reverse=reverse):
            print(f"  {name:<25} score {score:6.2f} | paper CI [{low:6.2f}, {high:6.2f}] | {diff:+.2f} vs mean | metric {runs_metrics[name]}")
        print()

    show("BETTER than paper CI", better, reverse=True)
    show("WORSE than paper CI", worse)
    show("WITHIN paper CI", within)

    if missing:
        print(f"=== No run found for: {', '.join(missing)} ===")

report_deviations(runs_scores, paper_nt_benchmark_scores, score_scale=100)

=== BETTER than paper CI: 0 ===

=== WORSE than paper CI: 0 ===

=== WITHIN paper CI: 0 ===

=== No run found for: H3, H3K4me1, H3K4me2, H3K4me3, H3K9ac, H3K14ac, H3K36me3, H3K79me3, H4, H4ac, enhancers, enhancers_types, promoter_all, promoter_no_tata, promoter_tata, splice_sites_acceptors, splice_sites_donors, splice_sites_all ===


In [6]:
paper_genomic_benchmark_scores = {
    "dummy_mouse_enhancers_ensembl": (0.784, 0.009),
    "human_enhancers_cohn": (0.743, 0.005),
    "human_enhancers_ensembl": (0.900, 0.004),
    "demo_coding_vs_intergenomic_seqs": (0.943, 0.001),
    "demo_human_or_worm": (0.967, 0.002),
    "human_ensembl_regulatory": (0.873, 0.002),
    "human_nontata_promoters": (0.951, 0.003),
    "human_ocr_ensembl": (0.793, 0.004),
}

runs = api.runs("pipe-verri-gh/Canonicalization", filters={
    "jobType": "genomic_benchmark",
    "group": "paper_baseline"
})

runs_scores, runs_metrics = build_runs_scores(runs, ["val/accuracy"])
report_deviations(runs_scores, paper_genomic_benchmark_scores, score_scale=1)

=== BETTER than paper CI: 1 ===
  dummy_mouse_enhancers_ensembl score   0.79 | paper CI [  0.78,   0.79] | +0.01 vs mean | metric val/accuracy

=== WORSE than paper CI: 5 ===
  human_ensembl_regulatory  score   0.70 | paper CI [  0.87,   0.88] | -0.18 vs mean | metric val/accuracy
  human_enhancers_ensembl   score   0.87 | paper CI [  0.90,   0.90] | -0.03 vs mean | metric val/accuracy
  human_nontata_promoters   score   0.92 | paper CI [  0.95,   0.95] | -0.03 vs mean | metric val/accuracy
  human_ocr_ensembl         score   0.79 | paper CI [  0.79,   0.80] | -0.01 vs mean | metric val/accuracy
  demo_coding_vs_intergenomic_seqs score   0.94 | paper CI [  0.94,   0.94] | -0.01 vs mean | metric val/accuracy

=== WITHIN paper CI: 2 ===
  human_enhancers_cohn      score   0.74 | paper CI [  0.74,   0.75] | -0.00 vs mean | metric val/accuracy
  demo_human_or_worm        score   0.97 | paper CI [  0.96,   0.97] | -0.00 vs mean | metric val/accuracy

